<div align="center">

# **SWE 485**  
## Customer Churn Prediction Using Machine Learning

</div>

---
---

<div align="center">

# **Phase 2 - Part B**

</div>



<div align="center">

# **Generative AI**

</div>


---
---

### **1. Introduction**


In this project, Google Gemini (gemini-2.5-flash) was selected as the generative AI model due to its fast response time, strong text generation capabilities, and ease of integration through a simple API. Additionally, it offers a free usage tier suitable for development and experimentation. This makes it suitable for generating recommendations based on model predictions.

In this part we will integrate the Generative AI into the customer churn prediction system to transform the model’s predictions into clear and actionable recommendations. Instead of only outputting whether a customer will stay or churn, the system generates meaningful insights based on the customer’s data and predicted outcome.

This is important because raw predictions alone are not sufficient for decision-making. By integrating Generative AI the system provides understandable and practical recommendations that help different stakeholders interpret the results and take appropriate actions.


#### **1.1. Objective**

The objective of this phase is to use Generative AI to:

- Transform model predictions into clear and understandable outputs  

- Provide actionable recommendations based on customer data and predicted outcome  

- Adapt recommendations to different stakeholders (customer, banker, analyst, marketing team)  

- Evaluate how different prompting techniques affect the quality of generated outputs


---
---

### **2. Generative AI Model Selection & Setup**
In this project, Google Gemini (gemini-2.5-flash) was selected as the generative AI model due to its fast response time, strong text generation capabilities, and ease of integration through a simple API. Additionally, it provides accessible usage options suitable for development and experimentation. This makes it suitable for generating recommendations based on model predictions.

To ensure secure and flexible usage the API key is stored using environment variables instead of being hardcoded in the code. The `.env` file is used to manage sensitive information safely, following best practices for key handling.

The setup process includes:
- Loading environment variables using the `dotenv` library, which allows sensitive data such as API keys to be securely stored and accessed without hardcoding them in the source code
- Retrieving the API key securely from the environment
- Initializing the Gemini client using the official SDK



In [57]:
import os
from dotenv import load_dotenv
from google import genai

load_dotenv()

gemini_api_key = os.getenv("GEMINI_API_KEY")
client = genai.Client(api_key=gemini_api_key)

---
---
### **3. Prompt Template Design Documentation**
in this section...

---

#### **3.1 Template 1**

---
#### **3.2 Template 2**

---

#### **3.3 Template 3**

---

#### **3.4 Template 4**

---
---

### **4. Implementation & API Integration Code**

The Generative AI functionality in this project is implemented as an end-to-end pipeline that connects user input, machine learning prediction, and prompt-based recommendation generation.

The process begins by securely configuring access to the Gemini API using environment variables. This ensures that sensitive credentials are not exposed in the source code and that the system can communicate with the generative model safely.

Next, the trained Random Forest pipeline - best model from phase 1 - is loaded from the saved file (`rf_pipeline.pkl`) using the `joblib` library. Loading the trained model instead of retraining it improves efficiency and allows the prediction stage to remain consistent with the supervised learning phase.

After that, the system collects customer information through runtime input forms. These forms allow new customer profiles to be entered directly in the notebook, making the workflow user interactive and suitable for testing different scenarios.

Since the machine learning model expects inputs in the same format used during training, the raw customer data is then passed through the same preprocessing and feature engineering steps applied in Phase 1. This includes feature creation, categorical encoding, log transformation, and alignment of the final columns with the exact training schema. This step is essential to ensure that the input is compatible with the trained model and that predictions remain valid.

Once preprocessing is complete, the Random Forest model generates predictions for each customer case. These predictions are then combined with the original user entered features to create prompt ready cases. This is an important design choice: the model uses the fully processed features internally for prediction, while Generative AI receives the original user features together with the predicted outcome as its input. This keeps the generated recommendations understandable and readable for human users.

Finally, these prompt-ready cases are passed into multiple prompt templates that apply different prompting techniques to different users.

Overall, this implementation creates a complete pipeline from raw runtime input to final Generative AI output, ensuring secure integration, model consistency, interpretability, and smooth interaction between the machine learning and generative components.

---

#### **4.1 User Input**

In this step, runtime input forms are created using the `ipywidgets` library so that customer information can be entered directly inside the notebook. This makes the workflow interactive and allows the system to accept new customer cases without retrieving from the dataset manually.

The cell defines a helper function called `make_case_form(case_num)`, which builds one complete input form for a single customer case. Each form contains all the raw input variables required before preprocessing, including:

- `CreditScore`
- `Geography`
- `Gender`
- `Age`
- `Tenure`
- `Balance`
- `NumOfProducts`
- `HasCrCard`
- `IsActiveMember`
- `EstimatedSalary`

Different widget types are used depending on the nature of the feature:
- `IntText` is used for integer-based values such as credit score, age, tenure, and number of products
- `FloatText` is used for continuous numeric values such as balance and estimated salary
- `Dropdown` is used for categorical or binary features such as geography, gender, card ownership, and activity status

This design is important because it ensures that users enter data in a structured and controlled format, which reduces input errors and improves consistency before preprocessing.

After defining the form structure, the cell creates three separate customer forms using:

`case_forms = [make_case_form(i) for i in range(1, 4)]`

This is done because the testing framework in this project evaluates the prompt templates using three different customer cases. Allowing three runtime inputs makes the process reproducible and directly aligned with the project evaluation design.

Finally, the forms are displayed vertically in the notebook using `widgets.VBox(...)`, so the user can fill in all three cases interactively before moving to the next step.

In summary this cell is responsible for collecting the raw customer input that will later be:
1. stored in a structured format,
2. preprocessed into model-ready features,
3. passed to the Random Forest model for prediction,
4. and finally combined with the prediction to form prompt-ready cases for Generative AI.

In [1]:
import ipywidgets as widgets
from IPython.display import display

def make_case_form(case_num):
    title = widgets.HTML(value=f"<h4>Case {case_num}</h4>")

    fields = {
        "CreditScore": widgets.IntText(description="CreditScore"),
        "Geography": widgets.Dropdown(
            options=["France", "Spain", "Germany"],
            description="Geography"
        ),
        "Gender": widgets.Dropdown(
            options=["Male", "Female"],
            description="Gender"
        ),
        "Age": widgets.IntText(description="Age"),
        "Tenure": widgets.IntText(description="Tenure"),
        "Balance": widgets.FloatText(description="Balance"),
        "NumOfProducts": widgets.IntText(description="NumProducts"),
        "HasCrCard": widgets.Dropdown(options=[0, 1], description="HasCrCard"),
        "IsActiveMember": widgets.Dropdown(options=[0, 1], description="IsActive"),
        "EstimatedSalary": widgets.FloatText(description="Salary")
    }

    form = widgets.VBox([title] + list(fields.values()))

    return {
        "widget": form,
        "fields": fields
    }

case_forms = [make_case_form(i) for i in range(1, 4)]

display(widgets.VBox([cf["widget"] for cf in case_forms]))



After the user fills in the input forms, this step extracts the entered values and converts them into a structured format that can be used in later stages of the pipeline.

The cell initializes an empty list called `cases_raw`, which will store all customer cases. It then iterates over each form in `case_forms`, where each form contains multiple input fields.

For each case:

- A dictionary (`case_data`) is created

- Each feature name (like CreditScore, Age, Balance) is used as a key

- The corresponding user-entered value is retrieved using `field.value`

- All features are grouped into a single dictionary representing one customer

Each constructed dictionary is then appended to the `cases_raw` list.

This process converts the interactive form inputs into a structured list of customer cases, making the data ready for preprocessing.

At the end of the cell, the collected cases are printed to verify correctness.

These cases represent different customer profiles and will be used throughout the remaining pipeline for preprocessing, prediction, and Generative AI evaluation.

In [36]:
cases_raw = []

for case_form in case_forms:
    case_data = {
        key: field.value
        for key, field in case_form["fields"].items()
    }
    cases_raw.append(case_data)

print("Collected Cases:")
for i, c in enumerate(cases_raw, 1):
    print(f"Case {i}: {c}")

Collected Cases:
Case 1: {'CreditScore': 416, 'Geography': 'France', 'Gender': 'Male', 'Age': 32, 'Tenure': 0, 'Balance': 0.0, 'NumOfProducts': 2, 'HasCrCard': 0, 'IsActiveMember': 1, 'EstimatedSalary': 874.0}
Case 2: {'CreditScore': 640, 'Geography': 'Spain', 'Gender': 'Male', 'Age': 45, 'Tenure': 1, 'Balance': 1298.0, 'NumOfProducts': 3, 'HasCrCard': 1, 'IsActiveMember': 0, 'EstimatedSalary': 33762.0}
Case 3: {'CreditScore': 770, 'Geography': 'France', 'Gender': 'Female', 'Age': 30, 'Tenure': 5, 'Balance': 10500.0, 'NumOfProducts': 1, 'HasCrCard': 1, 'IsActiveMember': 1, 'EstimatedSalary': 193880.0}


---

#### **4.2 Preprocessing**

In this step, the collected customer cases are transformed into a format that is compatible with the trained machine learning model. Since the Random Forest model was trained using a specific preprocessing pipeline in Phase 1, the same transformations must be applied here to ensure consistency and valid predictions.

First, the raw input cases (`cases_raw`) are converted into a Pandas DataFrame (`df_user`). This allows efficient manipulation of the data and application of vectorized operations.

The preprocessing process consists of four main stages:


**1. Feature Engineering**

New features are created to better represent customer behavior:

- `HasBalance`: Indicates whether the customer has a non-zero balance  
- `AgeGroup`: Categorizes customers into age ranges (like 30–39, 40–49)  
- `ActiveBalance`: Captures balance only when the customer is active  
- `CustomerValue`: Combines balance and number of products to estimate value  
- `InactiveHighBalance`: Identifies customers with high balance but low activity  

These features help capture patterns that are more meaningful for churn prediction compared to raw values alone.

**2. Categorical Encoding**

Categorical variables (`Geography`, `Gender`, `AgeGroup`) are converted into numerical format using one-hot encoding (`pd.get_dummies`).  

This is necessary because machine learning models require numerical inputs.  
The `drop_first=True` option is used to avoid redundancy and multicollinearity.

**3. Log Transformation**

The `CustomerValue` feature is transformed using a logarithmic function:

- `LogCustomerValue = log(1 + CustomerValue)`

This reduces the effect of extreme values and stabilizes the distribution, improving model performance.

**4. Column Alignment with Training Data**

The final step ensures that the input data has **exactly the same structure** as the data used during model training.

- A predefined list of `expected_columns` is used  
- Any missing columns are added with value `0`  
- The DataFrame is reordered to match the exact column order  

This step is critical because even small mismatches in feature order or presence can lead to incorrect predictions.

Finally, the preprocessed DataFrame (`df_user`) is displayed to verify that the transformation was applied correctly.

At this stage, the data is fully prepared and ready to be passed into the trained Random Forest model for prediction.

In [37]:
import pandas as pd
import numpy as np

# convert runtime input into DataFrame
df_user = pd.DataFrame(cases_raw)

# -----------------------------
# 1. feature engineering
# -----------------------------
df_user["HasBalance"] = (df_user["Balance"] > 0).astype(int)

age_bins = [17, 29, 39, 49, 59, 120]
age_labels = ["18_29", "30_39", "40_49", "50_59", "60_plus"]
df_user["AgeGroup"] = pd.cut(df_user["Age"], bins=age_bins, labels=age_labels)

df_user["ActiveBalance"] = df_user["Balance"] * df_user["IsActiveMember"]
df_user["CustomerValue"] = df_user["Balance"] * df_user["NumOfProducts"]
df_user["InactiveHighBalance"] = ((df_user["Balance"] > 0) & (df_user["IsActiveMember"] == 0)).astype(int)

# -----------------------------
# 2. categorical encoding
# -----------------------------
categorical_cols = ["Geography", "Gender", "AgeGroup"]
df_user = pd.get_dummies(df_user, columns=categorical_cols, drop_first=True, dtype=int)

# -----------------------------
# 3. log transform
# -----------------------------
df_user["LogCustomerValue"] = np.log1p(df_user["CustomerValue"].clip(lower=0))
df_user = df_user.drop(columns=["CustomerValue"])

# -----------------------------
# 4. match training columns exactly
# -----------------------------
expected_columns = [
    "CreditScore",
    "Age",
    "Tenure",
    "Balance",
    "NumOfProducts",
    "HasCrCard",
    "IsActiveMember",
    "EstimatedSalary",
    "HasBalance",
    "ActiveBalance",
    "InactiveHighBalance",
    "Geography_Germany",
    "Geography_Spain",
    "Gender_Male",
    "AgeGroup_30_39",
    "AgeGroup_40_49",
    "AgeGroup_50_59",
    "AgeGroup_60_plus",
    "LogCustomerValue"
]

for col in expected_columns:
    if col not in df_user.columns:
        df_user[col] = 0

df_user = df_user[expected_columns]

print("Preprocessed user cases:")
display(df_user)

Preprocessed user cases:


,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,HasBalance,ActiveBalance,InactiveHighBalance,Geography_Germany,Geography_Spain,Gender_Male,AgeGroup_30_39,AgeGroup_40_49,AgeGroup_50_59,AgeGroup_60_plus,LogCustomerValue
0,416,32,0,0.0,2,0,1,874.0,0,0.0,0,0,0,1,1,0,0,0,0.000000
1,640,45,1,1298.0,3,1,0,33762.0,1,0.0,1,0,1,1,0,1,0,0,8.267449
2,770,30,5,10500.0,1,1,1,193880.0,1,10500.0,0,0,0,0,1,0,0,0,9.259226


---

#### **4.3 Model Prediction**

In this step, the preprocessed customer cases are passed into the trained Random Forest model to generate churn predictions.

First, the saved model pipeline (`rf_pipeline.pkl`) is loaded using the `joblib` library.

The `joblib` library is a Python utility used for efficiently saving and loading machine learning models. During Phase 1 the trained Random Forest model was serialized (saved) into a `.pkl` file using `joblib.dump()`. This allows the model to be reused later without retraining.

In this step, the model is restored using:

`rf_model = joblib.load("rf_pipeline.pkl")`

This command reads the saved file from disk and reconstructs the trained model in memory, making it ready for immediate use in prediction.

Using `joblib` is important because:

- It preserves the exact trained model and its parameters

- It avoids the need to retrain the model every time the notebook runs

This allows the notebook to reuse the trained model directly without repeating the training process which enables a faster and more reliable pipeline, where predictions can be generated directly from new input data.

Next, the model predicts the outcome for each row in the preprocessed input DataFrame (`df_user`) using:

- `rf_model.predict(df_user)`

The model returns numerical predictions:
- `0` → customer is predicted to stay
- `1` → customer is predicted to churn

To make the results easier to understand and use in later prompt templates, these numerical outputs are converted into readable labels:
- `Will Stay`
- `Will Churn`

These labels are then added to the `df_user` DataFrame as a new column called `Prediction`, which allows the prediction result to remain linked to each customer case.

Finally, the prediction for each case is printed. This step is important because the predicted outcome becomes the main signal that will later be combined with the original customer features to create prompt-ready cases for Generative AI.

In [38]:
import joblib

# load trained model
rf_model = joblib.load("rf_pipeline.pkl")

# make predictions
preds = rf_model.predict(df_user)

# convert to labels
pred_labels = ["Will Stay" if p == 0 else "Will Churn" for p in preds]

# attach to dataframe
df_user["Prediction"] = pred_labels

print("Predictions:")
for i, p in enumerate(pred_labels, 1):
    print(f"Case {i}: {p}")

Predictions:
Case 1: Will Churn
Case 2: Will Churn
Case 3: Will Stay


---

#### **4.4 Building Prompt-Ready Cases**

In this step, the system prepares the final input that will be sent to the Generative AI model. Instead of passing raw model outputs alone, each customer case is structured to include both the original features and the predicted outcome.

First, an empty list called `cases` is initialized. The code then iterates over all collected cases using a loop.

For each customer case:
- The original user-entered features are taken from `cases_raw[i]`
- The corresponding prediction label is taken from `pred_labels[i]`
- A new dictionary is created with two keys:
  - `"features"` → contains the original customer data (readable format)
  - `"prediction"` → contains the model output (`Will Stay` or `Will Churn`)

Each combined dictionary is appended to the `cases` list.

This design is intentional and important:

- The **machine learning model** uses fully processed numerical data for accurate prediction  
- The **Generative AI model** uses the original human-readable features to generate clear recommendations  

By combining both into a single structure, we ensure that the generated responses are:
- context-aware (they include the prediction)
- interpretable (they use understandable feature values)
- aligned with real model outputs (not hypothetical labels)

These structured cases will be directly used in the next step to generate recommendations using different prompting techniques.

In [39]:
# build prompt-ready cases using:
# - raw user input for readability
# - model prediction from rf_model

cases = []

for i in range(len(cases_raw)):
    case = {
        "features": cases_raw[i],          # original user-entered features
        "prediction": pred_labels[i]       # model output
    }
    cases.append(case)

print("Prompt-ready cases:")
for i, case in enumerate(cases, 1):
    print(f"\nCase {i}:")
    print(case)

Prompt-ready cases:

Case 1:
{'features': {'CreditScore': 416, 'Geography': 'France', 'Gender': 'Male', 'Age': 32, 'Tenure': 0, 'Balance': 0.0, 'NumOfProducts': 2, 'HasCrCard': 0, 'IsActiveMember': 1, 'EstimatedSalary': 874.0}, 'prediction': 'Will Churn'}

Case 2:
{'features': {'CreditScore': 640, 'Geography': 'Spain', 'Gender': 'Male', 'Age': 45, 'Tenure': 1, 'Balance': 1298.0, 'NumOfProducts': 3, 'HasCrCard': 1, 'IsActiveMember': 0, 'EstimatedSalary': 33762.0}, 'prediction': 'Will Churn'}

Case 3:
{'features': {'CreditScore': 770, 'Geography': 'France', 'Gender': 'Female', 'Age': 30, 'Tenure': 5, 'Balance': 10500.0, 'NumOfProducts': 1, 'HasCrCard': 1, 'IsActiveMember': 1, 'EstimatedSalary': 193880.0}, 'prediction': 'Will Stay'}


---

#### **4.5 Prompting Techniques**

In this step, different prompting techniques are applied to evaluate how prompt design affects different factors of the generated recommendations.

Instead of using a single prompt style, multiple prompt templates are designed where each template represents a different prompting technique dediated to different user. All prompts receive the same input (customer features + model prediction), but differ in how the instruction is structured.

This allows a fair comparison of how each technique influences the generated output.

The following prompting techniques are implemented:


##### **4.5.1. One-shot Prompting (Customer)**

This technique provides the model with a single example before the actual task.  
The example demonstrates the expected structure and type of recommendations.

This helps guide the model to:
- produce clearer responses
- follow a consistent format
- generate more user-friendly recommendations

This technique is applied for **customer recommendations**, since end users benefit from guided and structured outputs.


In [76]:
def create_customer_prompt(case):

    return f"""
Example:

Customer data:
{{"CreditScore": 480, "Balance": 0, "IsActiveMember": 0}}

Prediction: Will Churn

Provide recommendations for the customer?

Recommendations:
1. Contact support for personalized financial advice
2. Increase account activity to avoid closure risk
3. Consider using more banking services

Now:

Customer data:
{case['features']}

Prediction: {case['prediction']}

Provide recommendations for the customer?
"""

In [77]:
print("===== CUSTOMER OUTPUT =====\n")

for i, case in enumerate(cases, 1):
    prompt = create_customer_prompt(case)

    response = client.models.generate_content(
        model="models/gemini-2.5-flash",
        contents=prompt
    )

    print(f"\n--- Case {i} ---")
    print(response.text)

===== CUSTOMER OUTPUT =====


--- Case 1 ---
Recommendations:
1.  Initiate immediate proactive outreach to understand the customer's initial experience and current needs, especially given their zero tenure and account balance.
2.  Offer personalized financial advice or resources for credit improvement, considering their very low credit score and estimated salary.
3.  Highlight the value and benefits of their existing products to encourage initial deposits and active account usage.

--- Case 2 ---
Recommendations:
1.  **Increase account activity:** Utilize digital banking services for regular transactions, bill payments, or direct deposits to become an active member and unlock full benefits.
2.  **Review your current products:** Schedule a personalized review to ensure your current 3 products meet your evolving needs and explore ways to consolidate your primary banking, potentially increasing your balance and engagement.
3.  **Connect with a financial advisor:** Reach out to a dedicated

---

##### **4.5.2. Zero-shot Prompting (Banker)**

This technique provides no examples and relies entirely on the model’s internal knowledge.

The model is directly instructed to generate recommendations based on the input.

This is suitable for **banking-related recommendations** where:
- the audience is more knowledgeable
- flexibility is preferred
- less rigid structure is acceptable
- more details are helpful


In [74]:
def create_banker_prompt(case):

    return f"""
Customer data:
{case['features']}

Prediction: {case['prediction']}

Provide recommendations for the banker regarding the customer?
"""

In [75]:
print("===== BANKER OUTPUT =====\n")

for i, case in enumerate(cases, 1):
    prompt = create_banker_prompt(case)

    response = client.models.generate_content(
        model="models/gemini-2.5-flash",
        contents=prompt
    )

    print(f"\n--- Case {i} ---")
    print(response.text)

===== BANKER OUTPUT =====


--- Case 1 ---
This customer presents a significant churn risk, especially considering their **Tenure is 0** (brand new customer) and they are predicted to churn immediately. The banker needs to act proactively and focus on building an immediate, positive relationship.

Here's a breakdown of the customer's profile and recommendations:

**Customer Profile Summary:**

*   **Very New:** Tenure is 0, meaning they've just joined the bank. This makes the churn prediction alarming – why would someone join and immediately leave?
*   **Financially Vulnerable:** Very low Credit Score (416) and Estimated Salary (874.0). This suggests potential financial difficulties or limited income.
*   **Low Engagement/Assets (Initially):** Zero Balance and no credit card with the bank yet. This makes it easy for them to leave as they have no significant ties.
*   **Younger:** Age 32, which means potential for a long-term relationship if nurtured correctly.
*   **Some Initial Engage

---

##### **4.5.3. Chain-of-Thought Prompting (Analyst)**

This technique includes an example that demonstrates step-by-step reasoning before giving recommendations.

The goal is to encourage the model to:
- analyze customer features
- explain why churn is likely
- generate recommendations based on that analysis

This is useful for **analysts**, who require deeper insights rather than direct actions.



In [72]:
def create_analyst_prompt(case):

    return f"""
Example:

Customer data:
{{"CreditScore": 480, "Balance": 0, "IsActiveMember": 0}}

Prediction: Will Churn

Provide recommendations for the banking analyst?

Reasoning:
The customer has a low credit score, indicating higher financial risk.
Zero balance and inactivity suggest weak engagement with the bank.
These factors increase the likelihood of churn.

Recommendations:
1. Prioritize this customer for retention monitoring.
2. Investigate low engagement drivers before offering new products.
3. Target interventions that increase account activity.

Now:

Customer data:
{case['features']}

Prediction: {case['prediction']}

Provide recommendations for the banking analyst?
"""

In [73]:
print("===== ANALYST OUTPUT =====\n")

for i, case in enumerate(cases, 1):
    prompt = create_analyst_prompt(case)

    response = client.models.generate_content(
        model="models/gemini-2.5-flash",
        contents=prompt
    )

    print(f"\n--- Case {i} ---")
    print(response.text)

===== ANALYST OUTPUT =====


--- Case 1 ---
Reasoning:
The customer presents an extremely high risk profile due to a very low credit score and very low estimated salary, indicating potential financial instability and limited capacity for traditional banking products.
Zero tenure and zero balance, despite having two products, suggest a critical failure in the initial onboarding process or immediate dissatisfaction, as the customer has not engaged financially with the bank at all.
The combination of high financial risk, lack of initial engagement, and very early churn signal (zero tenure) points to a customer who likely opened an account but found it unsuitable, was denied further services, or never fully committed to using the bank's offerings.

Recommendations:
1.  **Immediate Onboarding Intervention:** Initiate proactive contact with the customer to understand the specific barriers preventing account funding and active product usage, given the zero tenure and zero balance.
2.  **Re-ev

---

##### **4.5.4. Constraint-based Prompting (Marketing Team)**

This technique imposes explicit rules on the generated output, such as:
- limiting sentence length
- requiring feature-based justification
- enforcing diversity in recommendations

This ensures:
- structured outputs
- concise and customised recommendations
- consistency across responses

This is suitable for **marketing teams**, where outputs must follow specific formatting and communication guidelines to ensure consistency and clarity. Marketing teams often require structured and customizable content so they can tailor messages, campaigns, and strategies based on specific business needs.

In [70]:
def create_marketing_prompt(case):

    return f"""
Customer data:
{case['features']}

Prediction: {case['prediction']}

Provide recommendations for the marketing team?

Constraints:
- Each recommendation must be ONE sentence only
- All recommendations must be in a numbered list format
- Each must reference at least one customer feature
- Maximum 15 words per recommendation
- Recommendations must be different in purpose
- Do NOT use generic phrases like "improve service"
"""

In [71]:
print("===== MARKETING OUTPUT =====\n")

for i, case in enumerate(cases, 1):
    prompt = create_marketing_prompt(case)

    response = client.models.generate_content(
        model="models/gemini-2.5-flash",
        contents=prompt
    )

    print(f"\n--- Case {i} ---")
    print(response.text)

===== MARKETING OUTPUT =====


--- Case 1 ---
Here are marketing recommendations for this customer:

1.  Offer a personalized welcome incentive to boost engagement for new customers with zero tenure.
2.  Promote a special offer to encourage an initial deposit given their zero balance.
3.  Propose financial literacy resources to address the very low CreditScore.
4.  Introduce a secure card option, as they have no credit card and low CreditScore.
5.  Offer budget-friendly banking solutions suitable for their low estimated salary.

--- Case 2 ---
Here are recommendations for the marketing team:

1.  Offer exclusive benefits to reactivate this inactive member, fostering engagement.
2.  Proactively offer a loyalty incentive to retain this customer with 1-year tenure.
3.  Review and optimize the utility of this customer's 3 products for better value.

--- Case 3 ---
Here are some marketing recommendations:

1.  Offer exclusive wealth management options, given her high CreditScore and Estimat

---
---

### **5. Testing Framework & Output Comparison**

To evaluate the effectiveness of the designed prompt templates, a structured testing framework was implemented using three different customer cases entered at runtime. These cases were selected to represent varied customer profiles and prediction outcomes, allowing fair comparison across all prompt templates.

#### **Testing Framework**

Each prompt template was tested using the same 3 customer cases to ensure consistency and fairness in evaluation.  
For each case:

- Customer information was entered through runtime input forms
- The same preprocessing pipeline from Phase 1 was applied
- The trained Random Forest model generated the prediction
- Prompt-ready cases were constructed using the original customer features and the predicted outcome
- The same customer cases were passed to all prompt templates
- Responses were generated using the Gemini API

This approach ensures that all generated outputs are based on the same underlying customer data and actual model predictions, making the comparison fair and controlled.

#### **Output Comparison**

The generated outputs were analyzed using both **qualitative** and **quantitative** criteria.

##### **Qualitative Analysis**

Each prompt output will be evaluated based on the following factors:

- **Relevance:**  
  Whether the generated recommendations align with the predicted outcome and customer profile

- **Detail & Completeness:**  
  Whether the response provides enough useful and meaningful recommendation content

- **Clarity & Readability:**  
  Whether the output is easy to understand for the intended audience

- **Personalization:**  
  Whether the recommendations reflect the specific customer data rather than being generic

- **Safety & Factual Accuracy:**  
  Whether the response avoids unsupported assumptions, hallucinated details, or advice not grounded in the provided input

##### **Quantitative Comparison**

In addition to qualitative analysis, measurable indicators will be used to compare the outputs:

- **Response Length:**  
  Number of words in each generated output

- **Keyword Alignment with the Domain:**  
  Number of banking or retention  keywords appearing in the response

- **Readability Scores:**  
  Measured using readability indicators such as Flesch Reading Ease and Flesch-Kincaid Grade Level.  
  The Flesch Reading Ease score indicates how easy the text is to read (higher score = easier).  
  The Flesch-Kincaid Grade Level indicates the education level required to understand the text (like grade 8 = easy, grade 12 = more complex).

- **User Preference Simulation:**  
  This metric evaluates how well the generated output matches the needs and expectations of its intended stakeholder ( customer, banker, analyst, marketing team).

  The evaluation focuses on three key aspects:
  - **Simplicity:** How easy the output is to understand and use  
  - **Usefulness:** How actionable and relevant the recommendations are  
  - **Communication Style:** How well the tone, structure, and level of detail fit the target user  

  Each output is assessed from the user’s perspective, starting with its strengths and then identifying limitations.  
  A score out of 10 is assigned, where deductions are applied based on the impact of these limitations on the overall user experience.

This framework allows systematic comparison of all prompt templates and supports selecting the most effective prompting technique for recommendation generation.


---
#### **5.1. Prompt 1 Analysis (Customer – One-shot Prompting)**

##### **Qualitative Analysis**

- **Relevance:**  
  The recommendations are well aligned with both the predicted outcome and the customer profiles.  
  - Cases 1 and 2 (Will Churn) focus on engagement, activation, and financial guidance  
  - Case 3 (Will Stay) focuses on growth, investment, and premium opportunities  
  This shows that the model correctly adapts recommendations based on prediction and customer context.  

- **Detail & Completeness:**  
  The responses are consistent and well-structured, with exactly 3 recommendations per case, matching the provided example.  
  Each recommendation is actionable and includes brief justification.  
  However, the level of detail is moderate, and the recommendations are not deeply elaborated and sometimes lack specific actionable steps or concrete offers.

- **Clarity & Readability:**  
  The outputs are clear, concise, and easy to understand.  
  The language avoids heavy technical jargon and is suitable for customers.  
  A minor limitation is that some phrases are slightly formal ( “optimize your portfolio,” “financial advisory”), which may still require basic financial understanding.

- **Personalization:**  
  The recommendations are strongly personalized and reference customer-specific attributes such as credit score, tenure, balance, number of products, and estimated salary.  
  However, personalization is mostly at a surface level (feature referencing) rather than deeper behavioral insight or interaction between multiple features.

- **Safety & Factual Accuracy:**  
  The recommendations are mostly grounded in the provided data and prediction.  
  However, some mild assumptions are present such as suggesting customer intentions or needs (requiring financial advice or long-term planning) without explicit evidence in the input.


##### **Quantitative Analysis**

- **Response Length:**  
  - Case 1: 3 recommendations  
  - Case 2: 3 recommendations  
  - Case 3: 3 recommendations  
  → **Average: 3 recommendations per case (fully consistent)**  

- **Keyword Alignment with the Domain:**  
  Approximate domain-related keywords per case:  
  - Case 1: ~6 keywords  
  - Case 2: ~6 keywords  
  - Case 3: ~6 keywords  
  → **Average: ~6 domain keywords per response**  

- **Readability Scores:**  
  - Flesch Reading Ease: ~65–75 (fairly easy to read)  
  - Flesch-Kincaid Grade Level: ~8–10  
  → Indicates that the text is understandable for general users with basic education level, making it suitable for customer communication.  
  → Limitation: Some sentences still contain compound structures, which slightly increases the reading level and reduces simplicity.

- **User Preference Simulation:**  

  From the customer’s perspective, the output performs well because:
  - The recommendations are simple and easy to understand  
  - The language is clear and non-technical  
  - The tone is supportive and action-oriented, making the recommendations direct and immediately usable  

  However, there are some limitations:
  - The recommendations are quite general and not deeply personalized → -2  
  - No explanation is provided for why these actions are suggested → -1  

  - **User Preference Score:** 7/10


##### **Overall Observation**

With the updated model, Prompt 1 shows strong consistency, clarity, and alignment with the one-shot example. The technique successfully guides the model to produce structured and user-friendly outputs.

However, the recommendations remain moderately generic, with limited depth and some reliance on assumptions. While suitable for general customer guidance, the prompt could be improved by encouraging deeper personalization or more specific actionable insights.

---

#### **5.2. Prompt 2 Analysis (Banker — Zero-shot Prompting)**

##### **Qualitative Analysis**

- **Relevance:**  
  The recommendations are highly aligned with both the predicted outcome and the customer profiles.  
  - Cases 1 and 2 (Will Churn) focus on onboarding, engagement, and identifying churn causes  
  - Case 3 (Will Stay) focuses on relationship growth and maximizing customer value  
  This shows that the model correctly adapts its strategy based on both prediction and customer context, producing business-oriented recommendations.


- **Detail & Completeness:**  
  The responses are highly detailed and structured. Each case includes a profile summary, explanation of the situation, and multiple recommendations supported by rationale.  
  The recommendations are actionable and go beyond simple suggestions by explaining *why* and *how* actions should be taken.  

  However, the level of detail is very high which may reduce usability in scenarios requiring quick decision-making.


- **Clarity & Readability:**  
  The outputs are well-structured using headings, bullet points, and logical flow.  
  This improves readability for professional users such as bankers.  

  However, the responses are long and complex, and some sections require high domain knowledge to fully understand.

- **Personalization:**  
  The recommendations demonstrate strong personalization.  
  The model combines multiple features ( credit score, salary, tenure, activity level) to generate insights rather than simply restating inputs.  

  This results in deeper interpretation of customer behavior and more tailored recommendations.

- **Safety & Factual Accuracy:**  
  The recommendations are mostly grounded in the provided data and prediction.  
  The model avoids unrealistic or unsafe suggestions.  

  However, some assumptions are inferred  (financial vulnerability or customer intentions), which are not explicitly stated in the input data.


##### **Quantitative Analysis**

- **Response Length:**  
  - Case 1: ~550–650 words  
  - Case 2: ~500–600 words  
  - Case 3: ~550–700 words  
  - **Average:** ~550+ words per case  

  This indicates significantly detailed outputs.


- **Recommendation Count:**  
  - Case 1: 5 recommendations  
  - Case 2: 6–7 recommendations  
  - Case 3: 4 main groups with multiple sub-actions  
  - **Average:** ~5–6 recommendations per case  

  This shows higher coverage and depth of suggested actions.


- **Keyword Alignment with Domain:**  
  Approximate domain-related keywords per case:  
  - Case 1: ~10–14 keywords  
  - Case 2: ~12–16 keywords  
  - Case 3: ~14–18 keywords  
  - **Average:** ~13–16 keywords per response  

  This reflects strong alignment with banking and financial domain terminology.


- **Readability Scores:**  
  - Flesch Reading Ease: ~40–55  
  - Flesch-Kincaid Grade Level: ~11–13  

  → Indicates moderate-to-difficult readability  
  → Suitable for professional users rather than general customers  

  Limitation: Longer sentences and complex structure increase cognitive load.


- **User Preference Simulation:**  

  From the banker’s perspective, the output performs well because:
  - The recommendations are highly detailed and cover multiple business aspects  
  - The output provides strong reasoning that supports decision-making  
  - The tone is professional and analytical, aligning well with banking operations  

  However, there are some limitations:
  - The output is very long, reducing efficiency in time-sensitive environments → -2  
  - Some assumptions are inferred beyond explicit data → -1  

  - **User Preference Score:** 7/10


##### **Overall Observation**

Prompt 2 (Zero-shot prompting) produces highly detailed, and strategy-oriented outputs.  

This results in stronger performance in terms of insight generation, personalization, and domain alignment.

However, this comes with trade-offs in readability and conciseness, making it more suitable for professional users rather than general customers.

Overall, this prompt is most effective for scenarios where deep understanding and strategic recommendations are required.

---

#### **5.3. Prompt 3 Analysis (Analyst — Chain-of-Thought Prompting)**

##### **Qualitative Analysis**

- **Relevance:**  
  The recommendations are well aligned with both the predicted outcome and the customer profiles.  
  - Cases 1 and 2 (Will Churn) focus on identifying root causes such as low engagement, onboarding issues, financial instability, and weak consolidation  
  - Case 3 (Will Stay) focuses on stability, cross-selling, relationship deepening, and long-term value growth  
  This shows that the model not only reacts to the prediction but also explains *why* the outcome occurs before suggesting actions.

- **Detail & Completeness:**  
  The responses are detailed and well structured, with a clear separation between **Reasoning** and **Recommendations**.  
  Each case includes an explanation of the customer situation followed by targeted actions derived from that analysis.  

  The recommendations are actionable and logically supported.  
  Compared to Prompt 1, this prompt provides more analytical depth, and compared to Prompt 2, it is more focused and less excessively broad.

- **Clarity & Readability:**  
  The outputs are logically organized and easy to follow because the reasoning is explicitly separated from the recommendations.  
  This improves transparency and helps the reader understand how conclusions are formed.  

  However, the inclusion of reasoning makes the responses longer and slightly less concise than simpler prompts especially for users who only want direct recommendations.

- **Personalization:**  
  The recommendations show strong analytical personalization.  
  The model connects multiple features such as tenure, balance, activity, credit score, salary, and number of products to explain customer behavior rather than listing them individually.  

  This results in deeper, more context-aware recommendations and shows stronger feature interaction than simple referencing.

- **Safety & Factual Accuracy:**  
  The reasoning-based structure improves reliability by making the model’s logic visible.  
  Most conclusions are directly supported by the provided customer features and prediction.  

  However, a few interpretations are inferred rather than explicitly stated such as possible dissatisfaction, onboarding failure, or the customer finding the bank unsuitable. These are reasonable business inferences, but they are not directly observed facts from the input.

##### **Quantitative Analysis**

- **Response Length:**  
  - Case 1: ~230–300 words  
  - Case 2: ~140–180 words  
  - Case 3: ~170–220 words  
  - **Average:** ~190–230 words per case  

  This indicates moderately detailed outputs, balancing analytical depth and conciseness.

- **Recommendation Count:**  
  - Case 1: 4 recommendations  
  - Case 2: 3 recommendations  
  - Case 3: 4 recommendations  
  - **Average:** ~3.7 recommendations per case  

  This shows focused recommendations supported by explicit reasoning.

- **Keyword Alignment with Domain:**  
  Approximate domain-related keywords per case:  
  - Case 1: ~9–12 keywords  
  - Case 2: ~7–10 keywords  
  - Case 3: ~8–11 keywords  
  - **Average:** ~8–11 keywords per response  

  This reflects good alignment with banking, churn, and customer-retention terminology.

- **Readability Scores:**  
  - Flesch Reading Ease: ~50–60  
  - Flesch-Kincaid Grade Level: ~10–11  

  → Indicates moderate readability  
  → Easier than Prompt 2, but still more complex than Prompt 1 due to the reasoning section  

  Limitation: The additional explanation slightly increases sentence length and cognitive load.

- **User Preference Simulation:**  

  From the analyst’s perspective, the output performs well because:
  - The reasoning clearly explains how features lead to predictions  
  - The output links analysis to actionable recommendations  
  - The tone is analytical and structured, supporting interpretability  

  However, there are some limitations:
  - The reasoning adds complexity that may not always be necessary → -1  
  - Some interpretations extend slightly beyond the provided data → -1  

  - **User Preference Score:** 8/10

##### **Overall Observation**

Prompt 3 (Chain-of-Thought prompting) produces balanced outputs that combine reasoning and action.  
It improves interpretability by clearly showing how conclusions are derived from customer data before giving recommendations.

Compared to other prompts, it provides stronger explainability than Prompt 1 and better conciseness than Prompt 2. It also maintains strong personalization by connecting multiple customer features into a coherent interpretation.

Overall, this prompt is most effective for scenarios where understanding the reasoning behind the recommendation is as important as the recommendation itself.

---

#### **5.4. Prompt 4 Analysis (Marketing Team — Constraint-based Prompting)**

##### **Qualitative Analysis**

- **Relevance:**  
  The recommendations are generally aligned with both the predicted outcome and the customer profiles.  
  - Cases 1 and 2 (Will Churn) focus on engagement, incentives, activation, and retention-oriented offers  
  - Case 3 (Will Stay) focuses on product expansion, premium services, and loyalty-based rewards  
  This shows that the model adapts its recommendations according to both churn risk and customer value.

- **Detail & Completeness:**  
  The responses are concise and targeted, which matches the intended purpose of constraint-based prompting.  
  Each case provides short marketing-oriented suggestions rather than detailed explanations.  

  However, the output is less complete than the previous prompts because it sacrifices depth for brevity. Some recommendations are useful but not fully elaborated, and the rationale behind them is not always explicit.

- **Clarity & Readability:**  
  The outputs are very easy to read.  
  The recommendations are short, direct, and simple, which makes them highly accessible for quick review and campaign planning.  

  This is one of the strongest aspects of the prompt, as the constraints successfully improve readability and keep the recommendations concise.

- **Personalization:**  
  The recommendations show moderate personalization.  
  They reference customer-specific attributes such as:
  - zero tenure  
  - zero balance  
  - low credit score  
  - inactive membership  
  - high credit score  
  - high estimated salary  

However, personalization remains at a surface level as the imposed constraints prioritize concise and structured outputs over deeper analytical interpretation. This reflects a design trade-off rather than a model limitation.

- **Safety & Factual Accuracy:**  
  The outputs are mostly grounded in the provided input and avoid extreme or unsafe recommendations.  
  The suggestions stay within a realistic marketing and customer-engagement context.  

  However, a few recommendations are somewhat weakly supported such as suggesting “local French banking services tailored for her age group,” which is more speculative than directly grounded in the input data.

##### **Quantitative Analysis**

- **Response Length:**  
  - Case 1: 5 recommendations  
  - Case 2: 3 recommendations  
  - Case 3: 5 recommendations  
  - **Average:** ~4.3 recommendations per case  

  This indicates concise outputs overall, but also shows that the model did not consistently follow the intended number of recommendations.

- **Recommendation Count:**  
  - Case 1: 5 recommendations  
  - Case 2: 3 recommendations  
  - Case 3: 5 recommendations  
  - **Average:** ~4.3 recommendations per case  

  This shows moderate consistency, but weaker control compared to Prompt 1 where all cases followed the same count.

- **Keyword Alignment with Domain:**  
  Approximate domain-related keywords per case:  
  - Case 1: ~7–9 keywords  
  - Case 2: ~6–8 keywords  
  - Case 3: ~8–10 keywords  
  - **Average:** ~7–9 keywords per response  

  This reflects good alignment with marketing and banking terminology such as incentive, engagement, retention, product, loyalty, and wealth management.

- **Readability Scores:**  
  - Flesch Reading Ease: ~70–85  
  - Flesch-Kincaid Grade Level: ~6–8  

  → Indicates high readability  
  → Easier to understand than all previous prompts  

  This confirms that the constraint-based format successfully produced short and accessible recommendations.

- **User Preference Simulation:**  

  From the marketing team’s perspective, the output performs well because:
  - The recommendations are concise and easy to use quickly  
  - The outputs are suitable for direct campaign usage  
  - The tone is clear and action-focused, aligning with marketing needs  

  However, there are some limitations:
  - The recommendations lack strategic depth → -2  
  - The outputs may be too brief for long-term planning → -1  

  - **User Preference Score:** 7/10


##### **Overall Observation**

Prompt 4 (Constraint-based prompting) produces short, readable, and campaign-oriented outputs.  
Its main strength is clarity: the recommendations are simple, direct, and easy to scan, which is especially useful for marketing teams that often need concise and customizable ideas.

However, the strict constraints reduce analytical depth and lead to weaker consistency in following the requested format, especially in the number of recommendations. Compared to the other prompts this one is strongest in readability and weakest in depth.

Overall, this prompt is most effective for scenarios where concise, easily adaptable marketing suggestions are preferred over detailed explanation.

---
---

### **6. Best Prompt Selection & Justification**

---
---

### **7. Integration Plan for Final System**

---
---

### **8. Ethical Considerations & Limitations**